# E-commerce Sales Pipeline — Airflow + PySpark (runs for real, in the cloud)

This notebook installs a real Apache Airflow instance and runs the `ecommerce_sales_pipeline` DAG end-to-end, orchestrating the PySpark job in `jobs/process_orders.py`.

**Why Colab?** Airflow isn't officially supported on native Windows, and WSL2/Docker require admin rights that aren't available on this machine. Colab gives a real Linux environment for free, with no local install needed.

**Steps:**
1. Zip the `ecommerce-pipeline` project folder locally (instructions below).
2. Run the cells in order.
3. Upload the zip when prompted.
4. Watch the DAG run for real, with real logs.
5. Optionally open the full Airflow UI to take a screenshot for LinkedIn.

## Step 0 — Zip the project locally (run this on your PC, not in Colab)

Open PowerShell and run:

```powershell
Compress-Archive -Path "C:\Users\larissagomes\dev\ecommerce-pipeline\*" -DestinationPath "C:\Users\larissagomes\dev\ecommerce-pipeline.zip" -Force
```

You'll upload `ecommerce-pipeline.zip` in Step 2 below.

## Step 1 — Install Java, and create an isolated Python 3.11 environment for Airflow

Colab's own Python version changes over time (it's now 3.13), but Airflow 2.9.3 only ships dependency-constraint files up to Python 3.12 — installing it directly into Colab's Python fails. To avoid that entirely, this creates a separate Python 3.11 environment using `uv` (which downloads its own interpreter, independent of Colab's), and installs PySpark + Airflow there instead.

In [ ]:
import os

!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# Install uv (a fast Python package/environment manager) and have it fetch Python 3.11,
# independent of whatever Python version Colab itself is currently running.
!pip install -q uv
!uv python install 3.11

VENV_DIR = "/content/airflow-venv"
!uv venv {VENV_DIR} --python 3.11

VENV_PY = f"{VENV_DIR}/bin/python"
PY_VER = "3.11"

!uv pip install -q --python {VENV_PY} pyspark==4.2.0
!uv pip install -q --python {VENV_PY} "apache-airflow==2.9.3" --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-2.9.3/constraints-{PY_VER}.txt"

print("Install complete. Using venv Python:", VENV_PY)

## Step 2 — Upload and unzip the project

In [ ]:
from google.colab import files
import zipfile

print("Select ecommerce-pipeline.zip when prompted:")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
PROJECT_DIR = "/content/ecommerce-pipeline"
os.makedirs(PROJECT_DIR, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(PROJECT_DIR)

print("Project extracted to:", PROJECT_DIR)
!ls -la {PROJECT_DIR}

## Step 3 — Initialize Airflow and register the DAG

In [ ]:
import os

os.environ["AIRFLOW_HOME"] = "/content/airflow"
os.environ["ECOMMERCE_PROJECT_DIR"] = "/content/ecommerce-pipeline"
os.environ["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"

VENV_DIR = "/content/airflow-venv"
AIRFLOW_BIN = f"{VENV_DIR}/bin/airflow"

AIRFLOW_HOME = os.environ["AIRFLOW_HOME"]
dags_folder = os.path.join(AIRFLOW_HOME, "dags")
os.makedirs(dags_folder, exist_ok=True)

!cp /content/ecommerce-pipeline/dags/ecommerce_sales_dag.py {dags_folder}/

!{AIRFLOW_BIN} db migrate

!{AIRFLOW_BIN} dags list

## Step 4 — Run the DAG for real

`airflow dags test` executes every task in the DAG through Airflow's real execution engine (not a simulation) and prints each task's real logs — this is the standard, documented way to test a DAG end-to-end from the CLI.

In [ ]:
!{AIRFLOW_BIN} dags test ecommerce_sales_pipeline 2024-01-01

## Step 5 (optional) — Launch the full Airflow UI for a screenshot

This starts the real webserver + scheduler and opens the Airflow UI in a new tab via Colab's port proxy. Useful for a LinkedIn-ready screenshot of the DAG graph with green (successful) tasks.

Login with username `admin`, password `admin` (set in the next cell).

In [ ]:
!{AIRFLOW_BIN} users create --username admin --password admin --firstname Larissa --lastname Leal --role Admin --email admin@example.com || true

In [ ]:
import subprocess, time

# Launch standalone (webserver + scheduler) in the background.
standalone_proc = subprocess.Popen(
    [AIRFLOW_BIN, "standalone"],
    env=os.environ.copy(),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("Airflow standalone starting (PID:", standalone_proc.pid, ") — wait ~30s then run the next cell.")
time.sleep(30)

In [ ]:
from google.colab.output import serve_kernel_port_as_window

serve_kernel_port_as_window(8080)

In the Airflow UI: find `ecommerce_sales_pipeline`, click it, trigger a new run (the play button), and open the **Graph** view once it finishes — that's your screenshot.

When done, stop the background process:

In [ ]:
standalone_proc.terminate()
print("Airflow standalone stopped.")

## Step 6 — Inspect the pipeline output

In [ ]:
import pandas as pd

quality_dir = "/content/ecommerce-pipeline/data/processed/data_quality_report"
csv_file = [f for f in os.listdir(quality_dir) if f.endswith(".csv")][0]
print(pd.read_csv(os.path.join(quality_dir, csv_file)).to_string(index=False))

category_dir = "/content/ecommerce-pipeline/data/processed/sales_by_category"
csv_file = [f for f in os.listdir(category_dir) if f.endswith(".csv")][0]
print("\nSales by category:")
print(pd.read_csv(os.path.join(category_dir, csv_file)).to_string(index=False))